In [ ]:
!pip install -q elevenlabs google-genai lumaai openai pillow

In [ ]:
import base64

from IPython.display import display, Audio, Image
from PIL import Image as PillowImage
from google.colab import userdata
from pathlib import Path

In [ ]:
TITLE = "AeroStream"
SLOGANS = {
   "en": "Pure Sound. Zero Noise.",
   "de": "Reiner Klang. Kein Lärm.",
   "jp": "澄んだ音。雑音ゼロ。"
}
SCRIPTS = {
    "en": "Meet AeroStream, the next-generation noise-canceling headphones built for people who need clarity on demand. From focused work to late-night playlists, AeroStream delivers pure sound and shuts the city out.",
    "de": "Entdecken Sie AeroStream, die Noise-Cancelling-Kopfhörer für kompromisslose Klarheit im Alltag. Von konzentrierten Meetings bis zu Ihrem Lieblingssong liefert AeroStream reinen Klang und blendet die Welt um Sie herum aus.",
    "jp": "AeroStreamは、必要な瞬間にクリアな音へ集中できる次世代ノイズキャンセリングヘッドホンです。仕事の通話から音楽への没入まで、周囲の雑音を静かに抑え、澄んだサウンドだけを届けます。"
}

BRAND_STYLE = "premium, minimalist, brushed metal, soft shadow, futuristic consumer audio aesthetic"

## Generate images

In [ ]:
from google import genai
from google.genai import types

gemini_api_key = userdata.get('GEMINI_API_KEY')
gemini_client = genai.Client(api_key=gemini_api_key)

In [ ]:
def print_image_generation_response(response):
    print(f"Response id: {response.response_id}")
    print(f"Input tokens: {response.usage_metadata.prompt_token_count}; Output tokens: {response.usage_metadata.candidates_token_count}")

    for part in response.parts:
        if part.text is not None:
            if part.thought:
                print("[Thinking...]")
            else:
                print("Final response:")

            print(part.text)

        if part.inline_data is not None:
            format = part.inline_data.mime_type.split("/")[1]
            display(Image(data=part.inline_data.data, format=format))

def save_last_image(response: types.GenerateContentResponse, path_to_file: Path):
    last_image = next(p.inline_data for p in reversed(response.parts) if p.inline_data is not None)
    last_image.as_image().save(path_to_file)

### Generate base image (in English)

In [ ]:
base_image_prompt = f"""
                    Create a premium commercial hero image for high-end noise-canceling headphones.

                    Scene requirements:
                    - Subject: Photorealistic product photography with the headphones as the unmistakable focal point. Sleek aluminum body, soft-touch ear cushions, subtle reflections, and luxury commercial lighting.
                    - Brand Placement: The exact word "{TITLE}" must be cleanly and legibly printed on the flat outer shell of the ear cup. Use a simple, crisp, minimalist sans-serif font. The text must be perfectly formed, horizontally aligned, and completely free of clutter or distortion.
                    - Composition: Reserve the upper-left third of the image as clean, gradient negative space specifically for advertising copy.
                    - Slogan Text: Render the exact English phrase "{SLOGANS["en"]}" floating in the upper-left negative space. Typography must be bold, modern, sans-serif, premium, and perfectly legible.
                    - Background: A refined studio gradient with restrained depth and zero visual clutter.

                    Overall creative direction: {BRAND_STYLE}.
                    Crucial constraint: Ensure absolute perfect spelling for both "{TITLE}" and "{SLOGANS["en"]}". Do not add any extra logos, gibberish, or text anywhere else in the image.
                    """

In [ ]:
generate_base_image_response = gemini_client.models.generate_content(
    model="gemini-3.1-flash-image-preview",
    contents=[base_image_prompt],
    config=types.GenerateContentConfig(
        response_modalities=["text", "image"],
        thinking_config=types.ThinkingConfig(
            include_thoughts=True,
            thinking_level="high"
        ),
        image_config=types.ImageConfig(
            aspect_ratio="16:9",
            image_size="512px"
        )
    )
)

In [ ]:
print_image_generation_response(generate_base_image_response)
save_last_image(generate_base_image_response, "/content/base_image_en.png")

### Compare with OpenAI

In [ ]:
from openai import OpenAI

openai_api_key = userdata.get("OPENAI_API_KEY")
openai_client = OpenAI(api_key=openai_api_key)

In [ ]:
comparative_openai_response = openai_client.images.generate(
    model="gpt-image-1-mini",
    prompt=base_image_prompt,
    size="1536x1024",
    quality="high",
    output_format="png"
)

In [ ]:
for image in comparative_openai_response.data:
    image_bytes = base64.b64decode(image.b64_json)
    image_format = comparative_openai_response.output_format
    display(Image(data=image_bytes, format=image_format))

### Translate base image to German

In [ ]:
germany_edit_prompt = f"""
                      Edit the existing {TITLE} hero image for the German market.

                      Required changes:
                      - Replace the English slogan with the exact German slogan: "{SLOGANS["de"]}"
                      - Keep the typography bold, premium, modern, and highly legible
                      - Shift the headphones from metallic silver to a matte graphite finish
                      - Change the background to a minimalist studio setting
                      - Preserve the product angle, premium ad-photography quality, and landing-page hero framing

                      The result should feel designed for a German premium-tech campaign.
                      """

In [ ]:
generate_german_edit_response = gemini_client.models.generate_content(
    model="gemini-3.1-flash-image-preview",
    contents=[
        types.Content(role="user", parts=[types.Part(text=base_image_prompt)]),
        types.Content(role="model", parts=generate_base_image_response.parts),
        types.Content(role="user", parts=[types.Part(text=germany_edit_prompt)])
    ],
    config=types.GenerateContentConfig(
        response_modalities=["image"],
        thinking_config=types.ThinkingConfig(
            include_thoughts=False,
            thinking_level="minimal"
        ),
        image_config=types.ImageConfig(
            aspect_ratio="16:9",
            image_size="512px"
        )
    )
)

In [ ]:
print_image_generation_response(generate_german_edit_response)
save_last_image(generate_german_edit_response, "/content/base_image_de.png")

### Translate base image to Japanese

In [ ]:
japan_edit_prompt = f"""
                    Edit the original {TITLE} hero image for the Japanese market.

                    Required changes:
                    - Replace the English slogan with the exact Japanese slogan: "{SLOGANS["jp"]}"
                    - Keep the typography polished, intentional, and naturally integrated into the ad layout
                    - Shift the atmosphere toward a refined Tokyo night aesthetic with elegant neon reflections
                    - Add a subtle high-tech urban glow without making the product hard to read
                    - Preserve the premium product-photography quality and the clean landing-page composition

                    The result should feel modern, welcoming, cinematic, and premium.
                    """

In [ ]:
generate_japanese_edit_response = gemini_client.models.generate_content(
    model="gemini-3.1-flash-image-preview",
    contents=[
        japan_edit_prompt,
        PillowImage.open("/content/base_image_en.png"),
        PillowImage.open("/content/base_image_de.png")
    ],
    config=types.GenerateContentConfig(
        response_modalities=["image"],
        thinking_config=types.ThinkingConfig(
            include_thoughts=False,
            thinking_level="minimal"
        ),
        image_config=types.ImageConfig(
            aspect_ratio="16:9",
            image_size="512px"
        )
    )
)

In [ ]:
print_image_generation_response(generate_japanese_edit_response)
save_last_image(generate_japanese_edit_response, "/content/base_image_jp.png")

## Generate audios

In [ ]:
from elevenlabs import ElevenLabs

elevenlabs_api_key = userdata.get("ELEVENLABS_API_KEY")
elevenlabs_client = ElevenLabs(api_key=elevenlabs_api_key)

In [ ]:
def save_audio(audio_generator, path_to_file: Path):
    with path_to_file.open(mode="wb") as file:
        for chunk in audio_generator:
            file.write(chunk)


In [ ]:
generate_ad_speech_en_response = elevenlabs_client.text_to_speech.convert(
    model_id="eleven_flash_v2_5",
    voice_id="JBFqnCBsd6RMkjVDRZzb",
    text=SCRIPTS["en"],
    language_code="en"
)

save_audio(generate_ad_speech_en_response, Path("/content/ad_speech_en.mp3"))

In [ ]:
display(Audio(filename="/content/ad_speech_en.mp3"))

In [ ]:
generate_ad_speech_de_response = elevenlabs_client.text_to_speech.convert(
    model_id="eleven_flash_v2_5",
    voice_id="FTNCalFNG5bRnkkaP5Ug",
    text=SCRIPTS["de"],
    language_code="de"
)

save_audio(generate_ad_speech_de_response, Path("/content/ad_speech_de.mp3"))

In [ ]:
display(Audio(filename="/content/ad_speech_de.mp3"))

In [ ]:
generate_ad_speech_jp_response = elevenlabs_client.text_to_speech.convert(
    model_id="eleven_flash_v2_5",
    voice_id="3JDquces8E8bkmvbh6Bc",
    text=SCRIPTS["jp"],
    language_code="ja"
)

save_audio(generate_ad_speech_jp_response, Path("/content/ad_speech_jp.mp3"))

In [ ]:
display(Audio(filename="/content/ad_speech_jp.mp3"))

## Generate videos

In [ ]:
from lumaai import LumaAI

lumaai_api_key = userdata.get("LUMAAI_API_KEY")
lumaai_client = LumaAI(auth_token=lumaai_api_key)

In [ ]:
import time

def wait_until_completed(generation):
    for i in range(1000):
        if generation.state == "completed":
            return generation
        if generation.state == "failed":
            raise RuntimeError(f"Generation failed: {generation.failure_reason}")

        time.sleep(2)
        generation = lumaai_client.generations.get(id=generation.id)

    raise RuntimeError("Video generation couldn't finish in time.")

In [ ]:
video_prompt = """
               Create a cinematic 5-second product reveal for premium noise-canceling headphones.
               Start from the provided keyframe image. Keep the same product identity and typography.
               Use a sleek 360-degree micro-rotation of the headphones, subtle pulses of sound-wave light around the ear cups, and controlled camera motion.
               The style should feel polished, high-end, futuristic, and suitable for a global tech launch.
               """

In [ ]:
with Path("/content/base_image_en.png").open("rb") as file:
    keyframe_bytes = file.read()

# format_specifier looks like `data:image/png;base64,bytes...`
keyframe_data_url = f"data:image/png;base64,{base64.b64encode(keyframe_bytes).decode("utf-8")}"

In [ ]:
video_generation = lumaai_client.generations.create(
    model="ray-2",
    prompt=video_prompt,
    aspect_ratio="16:9",
    duration="5s",
    resolution="540p",
    loop=True,
    keyframes={
        "frame0": {
            "type": "image",

            # NOTE: Luma's Python SDK doesn't currently support uploading raw bytes for keyframes, so you need to provide a publicly accessible URL ending with the image name (and extension).
            "url": "..."
        }
    }
)

In [ ]:
wait_until_completed(video_generation)